In [3]:
import pandas as pd
import numpy as np
import fastf1 as fastf1
fastf1.Cache.enable_cache("../cache")
import fastf1.ergast as ergast
import datetime
import json
from fastf1.ergast import Ergast
import time


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import LinearSVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, make_scorer, confusion_matrix, mean_squared_error, mean_absolute_error

from xgboost import XGBRanker

from scipy.stats import randint, uniform

In [2]:
#years = [2018, 2019, 2020, 2021]
#years = [2022, 2023, 2024]
years = [2025, 2026]
all_events_schedules = []
for y in years:
    all_events_schedules.append(fastf1.get_event_schedule(y))
all_events =[]
all_events_schedules
for event_schedule in all_events_schedules:
    count = 27
    for i in range(1, count+1):
        try:    
            all_events.append(event_schedule.get_event_by_round(i))
        except:
            break

In [3]:
all_race_sessions = []
for event in all_events:
    
    all_race_sessions.append(event.get_session('R'))

In [4]:
#preload all session data
for race in all_race_sessions:
    race.load()

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Driver 4 completed the race distance 00:00.022000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INF

KeyboardInterrupt: 

In [ ]:
all_driver_results = []
race_details = []
temp_weather_data = []
for race in all_race_sessions:
    try:
        all_driver_results.append(race.results)
        race_details.append(race.session_info)
        temp_weather_data.append(race.weather_data)
    except:
        print(race.event)

RoundNumber                                                         15
Country                                                     Azerbaijan
Location                                                          Baku
OfficialEventName    FORMULA 1 QATAR AIRWAYS AZERBAIJAN GRAND PRIX ...
EventDate                                          2026-09-26 00:00:00
EventName                                        Azerbaijan Grand Prix
EventFormat                                               conventional
Session1                                                    Practice 1
Session1Date                                 2026-09-24 12:30:00+04:00
Session1DateUtc                                    2026-09-24 08:30:00
Session2                                                    Practice 2
Session2Date                                 2026-09-24 16:00:00+04:00
Session2DateUtc                                    2026-09-24 12:00:00
Session3                                                    Practice 3
Sessio

In [ ]:
#process the weather data to extract relevant information
weather_data = []
for weather in temp_weather_data:
    weather_data.append([weather['AirTemp'], weather['Rainfall'], weather['WindSpeed']])
for race_weather in weather_data:
    race_weather[0] = sum(race_weather[0])/len(race_weather[0])
    race_weather[2] = sum(race_weather[2])/len(race_weather[2])
    count_rain = 0
    for i in race_weather[1]:
        if i == True:
            count_rain += 1
    race_weather[1] = count_rain/len(race_weather[1])

In [ ]:
all_qualifying_sessions = []
for event in all_events:
    all_qualifying_sessions.append(event.get_session('Q'))

In [ ]:
for quali in all_qualifying_sessions:
    quali.load()

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '4'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '1', '63', '22', '23', '16', '44', '10', '55', '6', '14', '18', '7', '5', '12', '27', '30', '31', '87']
core           INFO 	Loading dat

RateLimitExceededError: any API: 500 calls/h

In [ ]:
all_driver_quali_results = []
quali_details = []

for quali in all_qualifying_sessions:
    try:
        all_driver_quali_results.append(quali.results)
        quali_details.append(quali.session_info)
    except:
        print(quali.event)

RoundNumber                                                         15
Country                                                     Azerbaijan
Location                                                          Baku
OfficialEventName    FORMULA 1 QATAR AIRWAYS AZERBAIJAN GRAND PRIX ...
EventDate                                          2026-09-26 00:00:00
EventName                                        Azerbaijan Grand Prix
EventFormat                                               conventional
Session1                                                    Practice 1
Session1Date                                 2026-09-24 12:30:00+04:00
Session1DateUtc                                    2026-09-24 08:30:00
Session2                                                    Practice 2
Session2Date                                 2026-09-24 16:00:00+04:00
Session2DateUtc                                    2026-09-24 12:00:00
Session3                                                    Practice 3
Sessio

In [ ]:
# for i in [46,45,44,43,42,41,40,39,38]:
#     del all_driver_quali_results[i]
# len(all_driver_quali_results)

In [ ]:
# for i in [41,40,39,38]:
#     del all_driver_results[i]
# len(all_driver_results)

38

In [ ]:
# organize the quali data 
organized_quali_data = []

def find_minimum_time(q1, q2, q3):
    times = [q1, q2, q3]
    min_time = None
    for time in times:
        if time is not None:
            if min_time is None or time < min_time:
                min_time = time
    return min_time

quali_event_count = 0
for quali_result in all_driver_quali_results:
    quali_driver_count = 0
    race_object = []
    while quali_driver_count < len(all_driver_quali_results[quali_event_count]["DriverNumber"]):
        q1 = quali_result.iloc[quali_driver_count]["Q1"]
        q2 = quali_result.iloc[quali_driver_count]["Q2"]
        q3 = quali_result.iloc[quali_driver_count]["Q3"]
        min_time = find_minimum_time(q1, q2, q3)

        driver_obj = []
        driver_obj.append(quali_details[quali_event_count]["Meeting"]["Key"])
        driver_obj.append(quali_result.iloc[quali_driver_count]["DriverId"])
        driver_obj.append(min_time)
        
        race_object.append(driver_obj)
        quali_driver_count += 1

    organized_quali_data.append(race_object)
    quali_event_count += 1

In [ ]:
# make quali time relative to the fastest time in that qualifying session
for i in range(len(organized_quali_data)):
    min_time = None
    for driver in organized_quali_data[i]:
        if driver[2] is not None:
            if min_time is None or driver[2] < min_time:
                min_time = driver[2]
    for driver in organized_quali_data[i]:
        if driver[2] is not None:
            driver[2] = driver[2] - min_time
        else:
            driver[2] = datetime.MAXYEAR 

In [ ]:
#corrects the NaT time to a max_time
organized_quali_data[37]
max_date = pd.Timedelta('2h 45m 59s') + pd.Timedelta(1, unit='s')
for quali_obj in organized_quali_data:
    for driver in quali_obj:
        if type(driver[2]) == pd._libs.tslibs.nattype.NaTType:
            driver[2] = max_date


/var/folders/vs/2gfyqkqs23xf81l6g9vwx2jw0000gn/T/ipykernel_46192/2265134125.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  max_date = pd.Timedelta('2h 45m 59s') + pd.Timedelta(1, unit='s')


In [ ]:
driver_list = []

event_count = 0
session_count = 1
year_curr = None

for driver_result in all_driver_results:
    race_driver_count = 0
    race_object = []
    
    
    while race_driver_count < len(all_driver_results[event_count]["DriverNumber"]): 

        if year_curr != race_details[event_count]["StartDate"].year:
            year_curr = race_details[event_count]["StartDate"].year
            session_count = 1
        driver_obj = []

        #0
        driver_obj.append(session_count) #adding round number
        #1
        driver_obj.append(race_details[event_count]["StartDate"].year)    
        #2
        driver_obj.append(race_details[event_count]["Meeting"]["Name"])
        #3
        driver_obj.append(race_details[event_count]["Meeting"]["Key"])
        #4
        driver_obj.append(race_details[event_count]["Key"])
        #5
        driver_obj.append(race_details[event_count]["Meeting"]["Circuit"]["Key"])
        #6
        driver_obj.append(race_details[event_count]["Meeting"]["Circuit"]["ShortName"])
        #7
        driver_obj.append(race_details[event_count]["StartDate"])    
        #8
        driver_obj.append(driver_result.iloc[race_driver_count]["DriverNumber"])
        #9
        driver_obj.append(driver_result.iloc[race_driver_count]["FullName"])
        #10
        driver_obj.append(driver_result.iloc[race_driver_count]["Abbreviation"])
        #11
        driver_obj.append(driver_result.iloc[race_driver_count]["DriverId"])
        #12
        driver_obj.append(driver_result.iloc[race_driver_count]["TeamId"])
        #13
        driver_obj.append(driver_result.iloc[race_driver_count]["Position"])
        #14
        driver_obj.append(driver_result.iloc[race_driver_count]["GridPosition"])
        #15
        driver_obj.append(driver_result.iloc[race_driver_count]["ClassifiedPosition"])
        #16
        driver_obj.append(driver_result.iloc[race_driver_count]["Status"])
        #17
        driver_obj.append(driver_result.iloc[race_driver_count]["Points"])
        #18
        driver_obj.append(driver_result.iloc[race_driver_count]["Time"])
        #19
        driver_obj.append(driver_result.iloc[race_driver_count]["Laps"])
        #20
        driver_obj.append(weather_data[event_count][0])
        #21
        driver_obj.append(weather_data[event_count][1])
        #22
        driver_obj.append(weather_data[event_count][2])
        #23
        #quali time added
        #24
        #drivers championship
        #25
        #constructors championship
        #26
        #EWMA of avg of past finish positions per team
        #27
        #EWMA of past finishing positions of driver
        #28
        #relevance rank

        race_object.append(driver_obj)
        race_driver_count += 1
    
    event_count += 1
    session_count += 1
    driver_list.append(race_object)


In [ ]:
#attach the quali data to the race data using meeting key and driver id

for race_object in driver_list:
    try:
        meeting_key = race_object[0][1] #get meeting key from the first driver in the race object as they are all same
        for quali_object in organized_quali_data:
            if race_object[0][3] == quali_object[0][0]: #compare meeting key
                for driver in race_object:
                    for quali_driver in quali_object:
                        if driver[11] == quali_driver[1]: #compare driver id
                            driver.append(quali_driver[2]) #append qualifying time
                    #if driver didnt have a quali time like the identifier(driverId) is nan
                    if len(driver) < 24:
                        driver.append(max_date)
    except:
        print(race_object)

#for i in driver_list: print(i)
                        

24
24
24
24
24
24


In [ ]:
#make race time relative to the fastest time in that race session
max_date = pd.Timedelta('2h 45m 59s') + pd.Timedelta(1, unit='s')
for race_obj in driver_list:
    for driver in race_obj:
        if type(driver[18]) == pd._libs.tslibs.nattype.NaTType:
            driver[18] = max_date

/var/folders/vs/2gfyqkqs23xf81l6g9vwx2jw0000gn/T/ipykernel_46192/2026265901.py:2: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  max_date = pd.Timedelta('2h 45m 59s') + pd.Timedelta(1, unit='s')


In [ ]:
#add driver and constructors championship data
ergast = Ergast()

def find_driver_points(driverId, driver_standings):
    driver_row = driver_standings.loc[driver_standings["driverId"] == driverId]

    if not driver_row.empty:
        return driver_row["points"].values[0]

    return 0.0

def find_constructor_points(teamId, constructor_standings):
    driver_row = constructor_standings.loc[constructor_standings["constructorId"] == teamId]

    if not driver_row.empty:
        return driver_row["points"].values[0]

    return 0.0

for session in driver_list:
    try:
        year = session[0][1]
        round = session[0][0]
        driver_standings = ergast.get_driver_standings(season=year,round=round).content[0]
        time.sleep(3.0)
        constructor_standings = ergast.get_constructor_standings(season=year,round=round).content[0]

        for driver in session:
            
            driverId = driver[11]
            teamId =  driver[12]
            
            driver_points = find_driver_points(driverId=driverId, driver_standings=driver_standings)
            driver.append(driver_points)

            constructor_points = find_constructor_points(teamId=teamId, constructor_standings=constructor_standings)
            driver.append(constructor_points)
    except:
        print(session)


Request for URL https://api.jolpi.ca/ergast/f1/2025/1/driverStandings.json failed; using cached response
Traceback (most recent call last):
  File "/Users/sri/Web-projects/F1 telemetry/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py", line 788, in urlopen
    response = self._make_request(
        conn,
    ...<10 lines>...
        **response_kw,
    )
  File "/Users/sri/Web-projects/F1 telemetry/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/Users/sri/Web-projects/F1 telemetry/.venv/lib/python3.14/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
  File "/opt/homebrew/Cellar/python@3.14/3.14.7/Frameworks/Python.framework/Versions/3.14/lib/python3.14/http/client.py", line 1478, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.7/Frameworks/Python.framework/Versions/3.14/lib/pytho

In [ ]:
import copy
copy_buffer = []
copy_buffer = copy.deepcopy(driver_list)


In [ ]:
#flatten the driver_list to create a single list of drivers with all their race and qualifying data
flat_driver_list_temp = []

for race_object in copy_buffer:
    for driver in race_object:
        flat_driver_list_temp.append(driver)


In [ ]:
columns_temp = ['Round', 'Year', 'RaceName', 'MeetingKey', 'SessionKey', 'CircuitKey', 'CircuitShortName', 'StartDate', 'DriverNumber', 'FullName', 'Abbreviation', 'DriverId', 'TeamId', 'Position', 'GridPosition', 'ClassifiedPosition', 'Status', 'Points', 'Time', 'Laps', 'AirTemp', 'RainFall', 'WindSpeed','QualiTime', 'DriverPoints', 'ConstructorPoints']

driver_temp_df = pd.DataFrame(flat_driver_list_temp, columns=columns_temp)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

#driver_temp_df.to_csv('driver_temp_results.csv', index=False)
driver_temp_df.to_csv('../data/driver_temp_results.csv', mode='a', header=False, index=False)

NameError: name 'flat_driver_list_temp' is not defined

In [ ]:
# fill empty position
read_df = pd.read_csv("../data/driver_temp_results.csv")

for col in ['Position']:
    read_df[col] = read_df[col].map(lambda x : 20.0 if pd.isna(x) else x)

In [46]:
#redo the flattening?

to_group = read_df.to_numpy().tolist()
curr_session = None
grouped = []
group = []

for row in to_group:
    sesh_key = row[4]

    if curr_session == None:
        curr_session = sesh_key

    elif curr_session != sesh_key:
        grouped.append(group)
        group = []
        curr_session = sesh_key

    group.append(row) 
if group:
    grouped.append(group)

driver_list = grouped

In [47]:
def calc_EWMA(prev_EWMA, alpha, curr_value):
    res = prev_EWMA * (1-alpha)
    res = alpha *  curr_value + res
    return res

In [48]:
def avg_constr_EWMA(race_idx, team):
    ewma = 0
    avg = 0
    prev_ewma = 0
    race = driver_list[race_idx]

    for driver in race:
        if driver[12] == team:
            avg += driver[13]
            prev_ewma = driver[26]
            
    ewma = calc_EWMA(prev_ewma, 0.56, avg)

    return ewma

In [49]:
# aggregated data of the constructors data(avg of the two drivers) with EWMA
for i in range(len(driver_list)):
    race = driver_list[i]

    #what about when it is the first race...?
    if i == 0:
        for j in range(len(race)):
            driver = race[j]
            driver.append(0)
        
    else:
        for j in range(len(race)):
            driver = race[j]
            team = driver[12]
            
            ewma_value = avg_constr_EWMA(i-1, team)
            driver.append(ewma_value)
            # for other_driver in race:
                
            #     if team == other_driver[12] and driverId != other_driver[11]:
            #         avg = driver[13] + other_driver[13]
            #         avg = avg/2

            # for k in range(i):
            #     for prev_driver in driver_list[k]:
            #         if team == prev_driver[12]:
            #             driver.append(calc_EWMA(prev_driver[26], alpha_constr_ewma, avg))
            #             break
            #     break
            # if len(driver) < 27:
            #     driver.append(avg)

In [50]:
#flatten the driver_list to create a single list of drivers with all their race and qualifying data
flat_driver_list = []

for race_object in driver_list:
    for driver in race_object:
        flat_driver_list.append(driver)

driver_list = flat_driver_list

In [54]:
def avg_driv_EWMA(driver_idx, driverId):
    ewma = 0
    prev_ewma = 0
    pt = 0
    i = driver_idx
    
    while i > 0:
        driver = driver_list[i]
        if driver[11] == driverId:
            prev_ewma = driver[27]
            pt = driver[13]
            break
        i -= 1
                
    ewma = calc_EWMA(prev_ewma, 0.56, pt)
    
    return ewma

In [55]:
# add aggregate data for each driver across all previous races
#alpha_driver_fin = 0.7
#whay if driver didnt exist before...????
#past finish position at this track
for i in range(len(driver_list)):
    driver1 = driver_list[i]

    idx = i - 1
    driver1.append(avg_driv_EWMA(idx, driver1[11]))

    #what is the effect of multiplying by 
    #if first appearence of driver
    # if len(driver1) < 28:
    #     driver_list[i].append(driver1[13])

In [56]:
#invert position to represent the relevance_score
for driver in driver_list:
    driver.append(23 - driver[13])

In [ ]:
columns = ['Round', 'Year', 'RaceName', 'MeetingKey', 'SessionKey', 'CircuitKey', 'CircuitShortName', 'StartDate', 'DriverNumber', 'FullName', 'Abbreviation', 'DriverId', 'TeamId', 'Position', 'GridPosition', 'ClassifiedPosition', 'Status', 'Points', 'Time', 'Laps', 'AirTemp', 'RainFall', 'WindSpeed','QualiTime', 'DriverPoints', 'ConstructorPoints', 'TeamAverageFinishEWMA','EWMAFinishPosition','relevance_score']

driver_df = pd.DataFrame(driver_list, columns=columns)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

driver_df.to_csv('../data/driver_results_final.csv', index=False)